# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Kaggle Notebook)
Panel phải: **Accelerator = GPU T4 x2**, **Internet = On**, **Add Input** dataset
`lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes`. Không cần Kaggle API key.

In [ ]:
import os, sys
!nvidia-smi -L
REPO_URL = "https://github.com/sown101/K4-Track-4-Day10-VisionLanguageModel.git"
REPO = "/kaggle/working/K4-Track-4-Day10-VisionLanguageModel"
if os.path.exists("/kaggle/working"):
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
    os.chdir(REPO)
print("cwd:", os.getcwd())
!pip -q install -r requirements-colab.txt

In [ ]:
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
!python -m pytest -q

## 1. Dữ liệu: tải và chia split

In [ ]:
# Dataset đã gắn qua Add Input; prepare_data tự tìm thư mục có images/ và labels-YOLO/
!ls /kaggle/input
!python scripts/prepare_data.py --source /kaggle/input

In [ ]:
import json
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from scripts.labcommon import CLASSES, read_jsonl
from lab.labels import yolo_to_pixel

DATA = Path("data")
splits = {s: read_jsonl(DATA / "splits" / f"{s}.jsonl") for s in ("train", "val", "test")}
meta = json.loads((DATA / "splits" / "meta.json").read_text(encoding="utf-8"))
print("dataset_version:", meta["dataset_version"])   # -> điền vào REPORT.md

# 1) Số ảnh / box từng lớp mỗi split (Câu 6)
print(f"{'split':6s} {'ảnh':>5s} {'box':>5s} " + " ".join(f"{c:>8s}" for c in CLASSES)
      + f" {'ảnh rỗng':>8s} {'phiên':>6s}")
for s, rows in splits.items():
    cnt = Counter(o["label"] for r in rows for o in r["objects"])
    empty = sum(1 for r in rows if not r["objects"])
    sess = len({r["session"] for r in rows})
    print(f"{s:6s} {len(rows):5d} {sum(cnt.values()):5d} " + " ".join(f"{cnt[c]:8d}" for c in CLASSES)
          + f" {empty:8d} {sess:6d}")

# 2) Không phiên quay nào nằm ở hai split (chống rò rỉ)
sess = {s: {r["session"] for r in rows} for s, rows in splits.items()}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    inter = sess[a] & sess[b]
    print(f"giao session {a}∩{b}: {len(inter)}")
    assert not inter, f"rò rỉ session giữa {a} và {b}: {sorted(inter)[:5]}"
print("sessions:", {s: sorted(v) for s, v in sess.items()})

# 3) Kích thước box (diện tích tương đối) theo lớp — để giải thích lớp nào khó (Câu 9)
for c in CLASSES:
    areas = sorted(o["yolo"][2] * o["yolo"][3] for r in splits["train"] for o in r["objects"]
                   if o["label"] == c)
    print(f"{c:8s} diện tích trung vị = {100 * areas[len(areas) // 2]:.2f}% khung hình (n={len(areas)})")

# 4) Vẽ 6 ảnh train có box
COLORS = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}

def draw_gt(rec, ax, title=None):
    img = Image.open(DATA / rec["image_path"]).convert("RGB")
    d = ImageDraw.Draw(img)
    for o in rec["objects"]:
        box = yolo_to_pixel(o["yolo"], rec["width"], rec["height"])
        d.rectangle(box, outline=COLORS[o["label"]], width=2)
        d.text((box[0] + 2, max(box[1] - 11, 0)), o["label"], fill=COLORS[o["label"]])
    ax.imshow(img); ax.axis("off")
    ax.set_title(title or rec["image_id"], fontsize=8)

samples = [r for r in splits["train"] if len({o["label"] for o in r["objects"]}) >= 2][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for rec, ax in zip(samples, axes.flat):
    draw_gt(rec, ax, f'{rec["image_id"]} ({len(rec["objects"])} box)')
plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("report/figures/gt_samples.png", dpi=110, bbox_inches="tight")
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab.labels import build_prompt, objects_to_answer, objects_to_boxes, box_to_pixel, yolo_to_pixel
from scripts.score import iou_xyxy, parse_output

PROMPT = build_prompt("norm1000")
print(PROMPT, "\n")

rec = next(r for r in splits["train"] if len(r["objects"]) >= 3)
ans = objects_to_answer(rec["objects"])
print(rec["image_id"], "->", ans)
assert parse_output(ans) is not None, "score.py không parse được đáp án!"

# Round-trip trên TOÀN BỘ box: YOLO -> 0–1000 (int) -> pixel, so với box gốc
max_err, min_iou, bumped, n = 0.0, 1.0, 0, 0
all_rows = [r for rows in splits.values() for r in rows]
for r in all_rows:
    W, H = r["width"], r["height"]
    assert parse_output(objects_to_answer(r["objects"], "norm1000", W, H)) is not None
    for o in r["objects"]:
        b = objects_to_boxes([o], "norm1000", W, H)[0]["bbox_2d"]
        px = box_to_pixel(b, "norm1000", W, H)
        xc, yc, w, h = o["yolo"]
        orig = ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)
        max_err = max(max_err, *(abs(p - q) for p, q in zip(px, orig)))
        min_iou = min(min_iou, iou_xyxy(px, orig))
        bumped += (round((xc + w / 2) * 1000) - round((xc - w / 2) * 1000) < 1
                   or round((yc + h / 2) * 1000) - round((yc - h / 2) * 1000) < 1)
        n += 1
print(f"{n} box | sai số round-trip lớn nhất = {max_err:.2f}px | IoU nhỏ nhất = {min_iou:.3f} "
      f"| box phải nới 1 đơn vị = {bumped}")
# Sai số > 0.5 px chỉ đến từ box bị kẹp vào [0,1000] (nhãn gốc vượt mép ảnh) hoặc box suy biến
odd = []
for s, rows in splits.items():
    for r in rows:
        for o in r["objects"]:
            W, H = r["width"], r["height"]
            orig = yolo_to_pixel(o["yolo"], W, H)
            px = box_to_pixel(objects_to_boxes([o], "norm1000", W, H)[0]["bbox_2d"], "norm1000", W, H)
            if max(abs(p - q) for p, q in zip(px, orig)) > 1 or iou_xyxy(px, orig) < 0.9:
                why = "suy biến (w/h = 0)" if o["yolo"][2] * o["yolo"][3] == 0 else "vượt mép ảnh → kẹp"
                odd.append((s, r["image_id"], o["label"], [round(v) for v in orig], why))
print(f"{len(odd)} box bất thường:", Counter((x[0], x[4]) for x in odd))
for x in odd[:10]:
    print("  ", x)

# Vẽ box đáp án (quy ngược về pixel) chồng lên box gốc: phải trùng nhau
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
draw_gt(rec, axes[0], "GT (YOLO)")
img = Image.open(DATA / rec["image_path"]).convert("RGB"); d = ImageDraw.Draw(img)
for b in json.loads(ans):
    d.rectangle(box_to_pixel(b["bbox_2d"], "norm1000", rec["width"], rec["height"]),
                outline=COLORS[b["label"]], width=2)
axes[1].imshow(img); axes[1].axis("off"); axes[1].set_title("JSON 0–1000 → pixel")
plt.tight_layout(); plt.show()

# Độ dài đáp án (ký tự) — để chọn max_new_tokens ở bước 3
lens = sorted(len(objects_to_answer(r["objects"])) for r in splits["train"])
print("độ dài đáp án: trung vị", lens[len(lens) // 2], "| p99", lens[int(.99 * len(lens))], "| max", lens[-1])

## 3. E0: zero-shot
Từ bước này cần GPU: **Session options → Accelerator = GPU T4 x2**. Mã suy luận nằm trong `lab/infer.py`
(nạp 4-bit NF4, giữ `visual` và `lm_head` ở fp16, pad trái, greedy). Với 2 GPU, test được chia đôi và chạy song song.

> **Lỗi đã gặp (transformers 5.19):** `llm_int8_skip_modules=["visual", ...]` *không* giữ ViT ở fp16, vì `should_convert_module` dùng `re.match` neo ở đầu tên module (`model.visual...`). Lần chạy E0 đầu tiên vì thế dùng ViT 4-bit (lưu lại ở `report/e0_vit4bit.json` để so sánh). `lab/infer.py` nay dùng `"model.visual"` và tự kiểm tra không còn `Linear4bit` trong visual.

In [ ]:
import torch
NGPU = torch.cuda.device_count()
print("GPU:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])
assert NGPU > 0, "Chưa bật GPU: Session options -> Accelerator = GPU T4 x2"

# Tải trọng số 1 lần trước, để các tiến trình con không cùng tải vào cache
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3-VL-4B-Instruct")

E0_DONE = Path("report/evidence/e0.json").exists()
if not E0_DONE:
    # Gỡ lỗi nhanh trên 8 ảnh val trước khi chạy cả test
    !python -m lab.infer --split val --limit 8 --out work/e0_val8.jsonl --max-new-tokens 384
    for r in read_jsonl("work/e0_val8.jsonl")[:5]:
        print(r["image_id"], "->", repr(r["raw"][:300]))

In [ ]:
# E0 trên toàn bộ test (311 ảnh). max_new_tokens=384: model gốc hay thêm khoảng trắng / lặp box
if not E0_DONE:
    !python -m lab.infer --split test --out work/e0.jsonl --max-new-tokens 384 --gpus {NGPU}
    !python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
else:
    print("đã có report/evidence/e0.json (chạy ngày trước), bỏ qua E0")

In [ ]:
# Phân tích lỗi E0 (Câu 2, Câu 3)
import re
from scripts.score import parse_output
from lab.viz import draw_pred_gt

ev = json.loads(Path("report/evidence/e0.json").read_text(encoding="utf-8"))
print({k: ev[k] for k in ("parse_valid_rate", "fp_per_image", "f1_ci95", "mean_iou")},
      "| F1 =", round(ev["micro"]["f1"], 4))
print({c: round(v["f1"], 3) for c, v in ev["per_class"].items()})

preds = {r["image_id"]: r["raw"] for r in read_jsonl("work/e0.jsonl")} if Path("work/e0.jsonl").exists() else {}
if preds:
    test = {r["image_id"]: r for r in splits["test"]}
    stats = Counter()
    for iid, raw in preds.items():
        p = parse_output(raw)
        stats["bọc ```"] += "```" in raw
        stats["có chữ ngoài JSON"] += bool(re.sub(r"```(json)?", "", raw).strip()[:1] not in "[")
        if p is None:
            stats["JSON hỏng / sai schema"] += 1
            continue
        stats["trả []"] += not p
        stats["nhãn ngoài 3 lớp"] += any(b["label"] not in CLASSES for b in p)
        stats["toạ độ > 1000"] += any(max(b["bbox"]) > 1000 for b in p)
        stats["số box dự đoán"] += len(p)
        stats["số box đúng"] += len(test[iid]["objects"])
    print(dict(stats))

    # Giả thuyết sai thang: chấm lại cùng output như thể là pixel (KHÔNG ghi vào report/evidence)
    !python scripts/score.py --pred work/e0.jsonl --name E0-as-pixel --coord pixel --out work/e0_as_pixel.json
    print("F1 nếu coi output là pixel:", json.loads(Path("work/e0_as_pixel.json").read_text())["micro"]["f1"])

    # 5 output thô + 6 ảnh minh hoạ (nét liền = dự đoán, nét đứt = đúng)
    ids = list(preds)[:: max(len(preds) // 6, 1)][:6]
    for iid in ids[:5]:
        print(iid, "->", repr(preds[iid][:300]))
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    for iid, ax in zip(ids, axes.flat):
        ax.imshow(draw_pred_gt(DATA / test[iid]["image_path"], test[iid], parse_output(preds[iid])))
        ax.axis("off"); ax.set_title(iid, fontsize=8)
    plt.tight_layout(); plt.savefig("report/figures/e0_examples.png", dpi=110, bbox_inches="tight"); plt.show()

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Mã trong `lab/train.py`: `encode_example()` (labels = -100 cho prompt và 220 token ảnh, chỉ học JSON đáp án + `<|im_end|>`),
`collate()` (right-pad `input_ids`/`attention_mask`/`labels`; `pixel_values`, `image_grid_thw` nối theo chiều 0).
Ô dưới chạy thử 5 bước E1 (`--check`): in số token ảnh, phần tính loss, số module LoRA khớp, số tham số huấn luyện.

In [ ]:
# Câu 1: tham số từng khối (đếm trên thiết bị meta; model 4-bit đóng gói nên numel() không đúng)
from lab.train import param_counts
for k, v in param_counts().items():
    print(f"{k:42s} {v:>15,}" if isinstance(v, int) and not isinstance(v, bool) else f"{k:42s} {v}")

# Kiểm tra bước 4–6 thật nhanh (5 bước) trước khi chạy 300 bước
if not Path("report/evidence/e1.json").exists():
    !CUDA_VISIBLE_DEVICES=0 python -m lab.train --config configs/e1.yaml --out work/dbg_e1 --max-steps 5 --check
    !CUDA_VISIBLE_DEVICES=0 python -m lab.train --config configs/e3.yaml --out work/dbg_e3 --max-steps 2 --check

## 5. Setup LoRA / đóng băng cho E1, E2, E3
`apply_setup()` trong `lab/train.py`: đóng băng tất cả → gradient checkpointing → LoRA r=16 trên
`language_model.layers.N.{self_attn.{q,k,v,o}_proj, mlp.{gate,up,down}_proj}` (252 module);
E3 thêm `visual.blocks.N.{attn.{qkv,proj}, mlp.linear_fc{1,2}}` (348 module).
E2/E3 mở `visual.merger` + `visual.deepstack_merger_list` ở **float32** (GradScaler cần master weight fp32).

Kiểm tra (kết quả ô trên): `252` / `348` module; tham số huấn luyện E2 − E1 ≈ 109M.

## 6–7. Huấn luyện, suy luận và chấm E1–E3
`lab/pipeline.py`: mỗi thí nghiệm = train 300 bước (AdamW 2e-4, warmup 5% + cosine, fp16 GradScaler, clip 1.0,
batch 2 × accum 4) → suy luận test (nạp adapter **và** `extra.pt` của projector) → `score.py --meta`.
Với T4 x2, hai thí nghiệm chạy song song (mỗi cái 1 GPU). Log từng thí nghiệm ở `work/<tên>.log`.

In [ ]:
!python -m lab.pipeline e1 e2

In [ ]:
!python -m lab.pipeline e3

In [ ]:
# Bảng tổng hợp E0–E3 (điền REPORT.md)
rows = []
for name in ["e0", "e1", "e2", "e3"] + sorted(p.stem for p in Path("report/evidence").glob("e4*.json")):
    p = Path("report/evidence") / f"{name}.json"
    if not p.exists():
        continue
    e = json.loads(p.read_text(encoding="utf-8"))
    m, pc = e.get("meta", {}), e["per_class"]
    rows.append([name.upper(), m.get("trainable_params", 0), m.get("peak_vram_gb", ""), m.get("train_minutes", ""),
                 round(e["micro"]["f1"], 3), [round(x, 3) for x in e["f1_ci95"]], round(e["fp_per_image"], 2),
                 round(e["parse_valid_rate"], 3)] + [round(pc[c]["f1"], 3) for c in CLASSES])
hdr = ["mã", "tham số", "VRAM", "phút", "F1", "CI95", "FP/ảnh", "valid", *CLASSES]
print(" | ".join(hdr))
for r in rows:
    print(" | ".join(f"{x:,}" if isinstance(x, int) else str(x) for x in r))

# Đường loss
fig, ax = plt.subplots(figsize=(8, 3.5))
for name in ["e1", "e2", "e3"]:
    p = Path("work") / name / "run.json"
    if p.exists():
        log = json.loads(p.read_text())["log"]
        ax.plot([x["step"] for x in log], [x["loss"] for x in log], label=name.upper())
ax.set_xlabel("bước"); ax.set_ylabel("loss"); ax.legend(); plt.tight_layout()
plt.savefig("report/figures/loss_curves.png", dpi=110); plt.show()

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
Đổi **đúng một biến** so với setup tốt nhất (sửa `EXPERIMENTS` trong `lab/pipeline.py` nếu setup tốt nhất không phải E1).

In [ ]:
# Ví dụ: ablation rank (chạy song song 2 GPU)
# !python -m lab.pipeline e4_rank_4 e4_rank_64

In [ ]:
# Gói bằng chứng + log nhỏ để tải về máy và commit (không gồm adapter/trọng số)
!cd /kaggle/working/K4-Track-4-Day10-VisionLanguageModel && zip -q -r /kaggle/working/results.zip report/evidence report/figures work/*.jsonl work/*.log work/*/run.json 2>/dev/null; ls -la /kaggle/working